In [1]:
pip install pandas numpy scikit-learn tensorflow fastapi uvicorn joblib seaborn matplotlib

In [2]:
import pandas as pd

df = pd.read_csv("clean_dataset_fixed.csv")

print(df.head())

                  date  product_id supplier_id supplier_name supplier_country  \
0  2023-01-01 00:00:00        1051     SUP_052  Supplier 052            Japan   
1  2023-01-01 01:00:00        1092     SUP_093  Supplier 093            Japan   
2  2023-01-01 02:00:00        1014     SUP_015  Supplier 015            India   
3  2023-01-01 03:00:00        1071     SUP_072  Supplier 072            India   
4  2023-01-01 04:00:00        1060     SUP_061  Supplier 061          Germany   

  supplier_region tier1_origin_port tier2_transit_port tier3_destination_port  \
0            Asia     Shanghai Port        Mumbai Port           Hamburg Port   
1            Asia  Los Angeles Port         Busan Port            Mumbai Port   
2            Asia  Los Angeles Port        Mumbai Port           Hamburg Port   
3            Asia        Busan Port       Hamburg Port       Los Angeles Port   
4          Europe     Shanghai Port         Busan Port           Hamburg Port   

   route_distance_km  ... 

In [21]:
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

print(df.columns)

Index(['date', 'product_id', 'supplier_id', 'supplier_name',
       'supplier_country', 'supplier_region', 'tier1_origin_port',
       'tier2_transit_port', 'tier3_destination_port', 'route_distance_km',
       'expected_time_hours', 'delay_hours', 'actual_time_hours',
       'port_congestion', 'supplier_lead_time', 'inventory_level',
       'safety_stock_level', 'units_sold', 'demand_volatility', 'weather_risk',
       'news_sentiment', 'fuel_price_index', 'business_unit',
       'product_category', 'priority_level', 'tier1_port_encoded',
       'tier2_port_encoded', 'tier3_port_encoded', 'delay_ratio',
       'time_efficiency'],
      dtype='object')


In [4]:
print(df.isnull().sum())

date                      0
product_id                0
supplier_id               0
supplier_name             0
supplier_country          0
supplier_region           0
tier1_origin_port         0
tier2_transit_port        0
tier3_destination_port    0
route_distance_km         0
expected_time_hours       0
delay_hours               0
actual_time_hours         0
port_congestion           0
supplier_lead_time        0
inventory_level           0
safety_stock_level        0
units_sold                0
demand_volatility         0
weather_risk              0
news_sentiment            0
fuel_price_index          0
business_unit             0
product_category          0
priority_level            0
dtype: int64


In [5]:
import numpy as np

df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna()

In [13]:
df["disruption_risk"] = df["disruption_risk"].replace({
    "Low": 0,
    "Medium": 1,
    "High": 2
})

KeyError: 'disruption_risk'

In [14]:
df["delay_ratio"] = df["delay_hours"] / df["actual_time_hours"]
df["time_efficiency"] = df["expected_time_hours"] / df["actual_time_hours"]

In [7]:
df["tier1_port_encoded"] = df["tier1_origin_port"].astype("category").cat.codes
df["tier2_port_encoded"] = df["tier2_transit_port"].astype("category").cat.codes
df["tier3_port_encoded"] = df["tier3_destination_port"].astype("category").cat.codes

In [15]:
features = [
    "shipment_delay_days",
    "port_congestion",
    "supplier_lead_time",
    "inventory_level",
    "units_sold",
    "demand_volatility",
    "weather_risk",
    "news_sentiment",
    "fuel_price_index",
    "route_distance_km",
    "tier_delay_factor",
    "expected_time_hours",
    "delay_hours",
    "actual_time_hours",
    "delay_ratio",
    "time_efficiency",
    "tier1_port_encoded",
    "tier2_port_encoded",
    "tier3_port_encoded"
]

In [17]:
missing = [col for col in features if col not in df.columns]
print("Missing Columns:", missing)

Missing Columns: ['shipment_delay_days', 'tier_delay_factor']


In [ ]:
X = df[features]
y = df["disruption_risk"]

print(X.shape)
print(y.shape)

KeyError: "['shipment_delay_days', 'tier_delay_factor'] not in index"

In [19]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

NameError: name 'X' is not defined

In [18]:
import numpy as np

print("NaN in X:", np.isnan(X).sum())
print("Inf in X:", np.isinf(X).sum())
print("NaN in y:", y.isnull().sum())

df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna()

NameError: name 'X' is not defined

In [ ]:
X = df[features]
y = df["disruption_risk"]
X = X.astype(float)
y = y.astype(int)

In [20]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

print(X_train.shape)
print(X_test.shape)

NameError: name 'X' is not defined

In [ ]:
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Dropout

model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(3, activation='softmax')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
print(X.dtypes)

shipment_delay_days    float64
port_congestion        float64
supplier_lead_time     float64
inventory_level        float64
units_sold             float64
demand_volatility      float64
weather_risk           float64
news_sentiment         float64
fuel_price_index       float64
route_distance_km      float64
tier_delay_factor      float64
expected_time_hours    float64
delay_hours            float64
actual_time_hours      float64
delay_ratio            float64
time_efficiency        float64
tier1_port_encoded     float64
tier2_port_encoded     float64
tier3_port_encoded     float64
dtype: object


In [ ]:
X = X.astype(float)

In [ ]:
print(df["disruption_risk"].unique())

[1 0 2]


In [ ]:
# Fix target mapping
df["disruption_risk"] = df["disruption_risk"].replace({
    "Low": 0,
    "Medium": 1,
    "High": 2
})

# Remove invalid rows
df = df.dropna(subset=["disruption_risk"])

# Convert to integer
y = df["disruption_risk"].astype(int)
print(y.unique())

[1 0 2]


In [ ]:
import numpy as np

print("NaN in X:", np.isnan(X).sum())
print("Inf in X:", np.isinf(X).sum())
print("NaN in y:", y.isnull().sum())

df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna()

NaN in X: shipment_delay_days    0
port_congestion        0
supplier_lead_time     0
inventory_level        0
units_sold             0
demand_volatility      0
weather_risk           0
news_sentiment         0
fuel_price_index       0
route_distance_km      0
tier_delay_factor      0
expected_time_hours    0
delay_hours            0
actual_time_hours      0
delay_ratio            0
time_efficiency        0
tier1_port_encoded     0
tier2_port_encoded     0
tier3_port_encoded     0
dtype: int64
Inf in X: shipment_delay_days    0
port_congestion        0
supplier_lead_time     0
inventory_level        0
units_sold             0
demand_volatility      0
weather_risk           0
news_sentiment         0
fuel_price_index       0
route_distance_km      0
tier_delay_factor      0
expected_time_hours    0
delay_hours            0
actual_time_hours      0
delay_ratio            0
time_efficiency        0
tier1_port_encoded     0
tier2_port_encoded     0
tier3_port_encoded     0
dtype: int64
NaN 

In [ ]:
X = df[features]
y = df["disruption_risk"]

In [ ]:
history = model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.2
)

Epoch 1/20
400/400 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.9113 - loss: 0.2578 - val_accuracy: 0.9625 - val_loss: 0.1143
Epoch 2/20
400/400 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.9516 - loss: 0.1332 - val_accuracy: 0.9634 - val_loss: 0.0878
Epoch 3/20
400/400 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9592 - loss: 0.1053 - val_accuracy: 0.9625 - val_loss: 0.0794
Epoch 4/20
400/400 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9656 - loss: 0.0905 - val_accuracy: 0.9794 - val_loss: 0.0546
Epoch 5/20
400/400 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9684 - loss: 0.0801 - val_accuracy: 0.9737 - val_loss: 0.0559
Epoch 6/20
400/400 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9725 - loss: 0.0696 - val_accuracy: 0.9812 - val_loss: 0.0469
Epoch 7/20
400/400 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.9758 - loss: 0.0596 - val_accuracy: 0.9797 - val_loss: 0.0518
Epoch 8/20
400/400 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9788 - loss: 0.0552 - val_accuracy: 0.